# Point Cloud and Compression-Grid Bifiltration Visualization

This notebook visualizes point-cloud examples and scc2020 chain-complex generators on the compressed finite grid. It does not call 2pac or GUDHI.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
from pprint import pprint

from bifiltration_visualization import (
    describe_pointcloud,
    discover_pointcloud_files,
    discover_scc2020_files,
    generator_records,
    load_compressed_chain_complex,
    load_pointcloud_for_visualization,
    plot_alpha_complex_at_grid,
    plot_alpha_complex_grid,
    plot_pointcloud,
    reconstruct_geometric_simplices,
    selected_grid_coordinates,
    summarize_compressed_complex,
)
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
POINTCLOUD_DIR = PROJECT_ROOT / "data" / "pointcloud_examples"
SCC_DIR = PROJECT_ROOT / "data" / "chain_complex_scc2020"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("POINTCLOUD_DIR:", POINTCLOUD_DIR)
print("SCC_DIR:", SCC_DIR)

## 2. Manual Paired File Selection

Set `POINTCLOUD_PATH` and `SCC_PATH` as a matched pair. File discovery below is only a reference list.

In [ ]:
pointcloud_files = discover_pointcloud_files(POINTCLOUD_DIR)
scc_files = discover_scc2020_files(SCC_DIR)

print("Point-cloud files:")
for index, path in enumerate(pointcloud_files):
    print(f"  [{index}] {path.relative_to(PROJECT_ROOT)}")

print("\nscc2020 files:")
for index, path in enumerate(scc_files):
    print(f"  [{index}] {path.relative_to(PROJECT_ROOT)}")

# Manual paired selection. Change both paths together.
POINTCLOUD_PATH = POINTCLOUD_DIR / "two_circles_200pts_knn_function.txt"
SCC_PATH = SCC_DIR / "two_circles_200pts_knn_function.alpha.scc2020.txt"

print("\nSelected point cloud:", POINTCLOUD_PATH)
print("Selected scc2020:", SCC_PATH)

## 3. Point Cloud Visualization

Set `POINTCLOUD_MODE` to `auto`, `xy`, `xyz`, `xyf`, or `xyzf` if the automatic format choice is not what you want.

In [ ]:
POINTCLOUD_MODE = "auto"

try:
    pointcloud = load_pointcloud_for_visualization(POINTCLOUD_PATH, mode=POINTCLOUD_MODE)
    pprint(describe_pointcloud(pointcloud))
    plot_pointcloud(pointcloud)
    plt.show()
except Exception as exc:
    pointcloud = None
    print(f"Point-cloud visualization skipped: {exc}")

## 4. Load and Compress scc2020 Chain Complex

In [ ]:
try:
    compressed_data = load_compressed_chain_complex(SCC_PATH)
    records = generator_records(compressed_data.complex, compressed_data.compressed_complex)
    geometric_records = reconstruct_geometric_simplices(
        compressed_data.complex,
        compressed_data.compressed_complex,
    )
    pprint(summarize_compressed_complex(compressed_data))
    print("\nReconstructed geometric generators:", len(geometric_records))
    if pointcloud is not None:
        c0_count = len(compressed_data.complex.term_grades[0])
        if pointcloud.point_count < c0_count:
            raise ValueError(
                f"The paired point cloud has {pointcloud.point_count} points, but C0 has {c0_count} generators."
            )
    print("\nValidation valid:", compressed_data.validation.is_valid)
    print("Validation warnings:", compressed_data.validation.warnings)
    print("Validation errors:", compressed_data.validation.errors)
    print("Final grade block:", compressed_data.validation.has_final_grade_block)
except Exception as exc:
    compressed_data = None
    records = []
    geometric_records = []
    print(f"scc2020 loading skipped: {exc}")

## 5. Select Compression Grid Grades

The full compression grid uses integer coordinates from `(0, 0)` to `compression.upper`. If the grid is large, this notebook shows the first and last two coordinates on each axis, plus four uniformly random middle coordinates. `GRID_VERTICAL_SPACING = None` uses the automatic compact vertical spacing; set it to a float to override. `SHOW_ALL_POINTCLOUD_POINTS = False` means nonexistent vertices are hidden by default.

In [ ]:
GRID_EDGE_COUNT = 2
GRID_RANDOM_MIDDLE_COUNT = 4
GRID_RANDOM_SEED = None  # Use an integer for reproducible random middle grades.
GRID_VERTICAL_SPACING = None  # None uses the automatic compact default; try 0.04 or 0.12 manually.
SHOW_ALL_POINTCLOUD_POINTS = False  # Set True to show the full point cloud as a gray background.

if compressed_data is not None:
    X_GRID_COORDINATES, Y_GRID_COORDINATES = selected_grid_coordinates(
        compressed_data.compression,
        edge_count=GRID_EDGE_COUNT,
        middle_count=GRID_RANDOM_MIDDLE_COUNT,
        random_seed=GRID_RANDOM_SEED,
        max_full_cells=100,
    )
    print("Compression upper:", compressed_data.compression.upper)
    print("x coordinates shown:", X_GRID_COORDINATES)
    print("y coordinates shown:", Y_GRID_COORDINATES)
    print("edge coordinates per side:", GRID_EDGE_COUNT)
    print("random middle coordinates per axis:", GRID_RANDOM_MIDDLE_COUNT)
    print("random seed:", GRID_RANDOM_SEED)
    print("vertical subplot spacing:", GRID_VERTICAL_SPACING if GRID_VERTICAL_SPACING is not None else "auto")
    print("show full point-cloud background:", SHOW_ALL_POINTCLOUD_POINTS)
    print("x axis is the compressed radius/alpha coordinate.")
    print("y axis is the compressed function-value coordinate.")
else:
    X_GRID_COORDINATES, Y_GRID_COORDINATES = [], []
    print("No compressed chain complex loaded.")

## 6. Alpha Complex Panels on the Compression Grid

Each panel corresponds to one compressed grade `(i, j)`. It draws every reconstructed simplex whose compressed birth grade is `<= (i, j)` coordinatewise. Dark vertices, blue edges, and filled triangles are the currently present complex. If `SHOW_ALL_POINTCLOUD_POINTS = True`, gray dots show the full paired point cloud as optional background.

In [ ]:
if compressed_data is None:
    print("No compressed chain complex loaded.")
elif pointcloud is None:
    print("No paired point cloud loaded.")
else:
    try:
        plot_alpha_complex_grid(
            pointcloud,
            geometric_records,
            compressed_data.compression,
            x_coordinates=X_GRID_COORDINATES,
            y_coordinates=Y_GRID_COORDINATES,
            vertical_spacing=GRID_VERTICAL_SPACING,
            show_all_points=SHOW_ALL_POINTCLOUD_POINTS,
            title="Alpha complex at selected compression-grid grades",
        )
        plt.show()
    except Exception as exc:
        print(f"Alpha-complex grid visualization skipped: {exc}")

## 7. Single Selected Grade

Set `SELECTED_GRID_GRADE` below, then rerun this cell to inspect one grade at a larger size.

In [ ]:
SELECTED_GRID_GRADE = (641, 110)

if compressed_data is None:
    print("No compressed chain complex loaded.")
elif pointcloud is None:
    print("No paired point cloud loaded.")
else:
    try:
        fig, ax = plt.subplots(figsize=(6, 6))
        plot_alpha_complex_at_grid(
            ax,
            pointcloud,
            geometric_records,
            SELECTED_GRID_GRADE,
            show_all_points=SHOW_ALL_POINTCLOUD_POINTS,
            show_labels=True,
        )
        ax.set_title(f"Alpha complex at compressed grade {SELECTED_GRID_GRADE}")
        plt.show()

        present = [
            record
            for record in geometric_records
            if record.compressed_grade[0] <= SELECTED_GRID_GRADE[0]
            and record.compressed_grade[1] <= SELECTED_GRID_GRADE[1]
        ]
        by_dim = {dim: sum(1 for record in present if record.dimension == dim) for dim in sorted({r.dimension for r in present})}
        print("Present simplex counts by dimension:", by_dim)
    except Exception as exc:
        print(f"Selected-grade visualization skipped: {exc}")